# Global PCA for Industrial Anomaly Detection

**CS 663 — Digital Image Processing | MVTec AD**

Each complete grayscale image is flattened into one feature vector. PCA learns the dominant appearance patterns from normal training images. New images are projected onto the learned subspace and reconstructed. Pixels with large squared reconstruction errors are treated as possible defects.

The model is fitted using defect-free training images only. Normal validation images are used to set the image-level threshold. The official test split and its masks are reserved for final evaluation.

The PCA calculations are implemented directly using NumPy matrix operations and eigenvalue decomposition. No prebuilt PCA estimator is used.

## 1. Setup and category selection

Set RUN_ALL_CATEGORIES to True to run the full 15-category benchmark. Leave it False for a quick smoke test on bottle. The dataset must be available under the configured MVTec root.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from scipy.ndimage import label as connected_components
from scipy.integrate import trapezoid
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/CS-663")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "global-pca-lecture-review-v2"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "global-pca-lecture-review-v2"], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "global-pca-lecture-review-v2"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", "global-pca-lecture-review-v2", "https://github.com/pranitas-cloud/CS-663.git", str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / "src").is_dir():
        raise RuntimeError("Open this notebook from the CS-663 repository root.")

os.chdir(REPO)
if str(REPO.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO.resolve()))

from src.data.pipelines import load_config, build_dataset, stack_images

CONFIG = load_config("configs/default.yaml")
DEFAULT_DATA_ROOT = Path("/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection") if IN_COLAB else REPO / "data" / "mvtec_anomaly_detection"
DATA_ROOT = Path(os.environ.get("MVTEC_DATA_ROOT", DEFAULT_DATA_ROOT)).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO / DATA_ROOT
CONFIG["data"]["root"] = str(DATA_ROOT)

SEED = int(CONFIG.get("seed", 42))
IMAGE_SIZE = int(CONFIG["pca"].get("image_size", 256))
EXPECTED_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]
RUN_ALL_CATEGORIES = False
SMOKE_TEST_CATEGORY = "bottle"

np.random.seed(SEED)
torch.manual_seed(SEED)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"MVTec folder not found: {DATA_ROOT}")

available_categories = {
    folder.name for folder in DATA_ROOT.iterdir()
    if folder.is_dir() and (folder / "train" / "good").is_dir()
}

if RUN_ALL_CATEGORIES:
    missing_categories = sorted(set(EXPECTED_CATEGORIES) - available_categories)
    if missing_categories:
        raise FileNotFoundError(f"Missing MVTec categories: {missing_categories}")
    CATEGORIES_TO_RUN = EXPECTED_CATEGORIES
else:
    if SMOKE_TEST_CATEGORY not in available_categories:
        raise FileNotFoundError(f"Category not found: {SMOKE_TEST_CATEGORY}")
    CATEGORIES_TO_RUN = [SMOKE_TEST_CATEGORY]

print("Repository:", REPO)
print("Dataset:", DATA_ROOT)
print("Categories to run:", CATEGORIES_TO_RUN)
print("Run all 15 categories:", RUN_ALL_CATEGORIES)
print("Image size:", IMAGE_SIZE)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

## 2. PCA method

Each normal training image is resized, converted to grayscale and flattened into a vector in R^D. The training mean is

$$
\mu = \frac{1}{N}\sum_{i=1}^{N}x_i,
\qquad X_c = X - \mathbf{1}\mu^T.
$$

The lecture defines the covariance using the uncorrected maximum-likelihood estimate

$$
C = \frac{1}{N}X_c^T X_c.
$$

The principal directions are the eigenvectors of C, ordered by decreasing eigenvalue. They are the directions of maximum variance described in the PCA lecture (relevant slides 8–17). We use an economy SVD, X_c = UΣV^T, to obtain the same directions without explicitly constructing the very large D×D covariance matrix:

$$
C V = V\frac{\Sigma^2}{N}.
$$

This is a computational implementation of the lecture's covariance eigendecomposition, not a different PCA objective. SVD is the numerical extension used because each image has 65,536 pixel features while the number of training images is much smaller.

For a retained basis V_k, projection and reconstruction are

$$
z = V_k^T(x-\mu), \qquad \hat{x}=\mu+V_kz.
$$

We retain the smallest number of components explaining at least 95% of the training variance. The pixel anomaly map is the squared reconstruction residual, (x−x̂)^2. Using its maximum as the image score and calibrating an image threshold on normal validation images are task-specific choices for anomaly detection, not PCA equations stated in the lecture.

In [ ]:
from src.pca.global_pca import GlobalPCA

## 3. Synthetic sanity check

This small test checks output dimensions, finite non-negative anomaly scores and reconstruction behavior before the real dataset is processed.

In [ ]:
random_generator = np.random.default_rng(SEED)
toy_images = random_generator.normal(size=(8, 1, 16, 16)).astype(np.float32)

toy_model = GlobalPCA(n_components="all", method="svd", ddof=0).fit(toy_images)
toy_reconstruction = toy_model.reconstruct(toy_images)
toy_anomaly_map = toy_model.anomaly_map(toy_images)

assert toy_reconstruction.shape == toy_images.shape
assert toy_anomaly_map.shape == (8, 16, 16)
assert np.isfinite(toy_anomaly_map).all()
assert toy_anomaly_map.min() >= 0
assert np.mean((toy_images - toy_reconstruction) ** 2) < 1e-10

print("Global PCA synthetic checks passed.")

## 4. Evaluation protocol

The PCA mean and basis are fitted on train/good only. The deterministic validation split also contains only normal images. The image threshold is the 95th percentile of validation image scores, corresponding to a nominal 5% validation false-positive rate. Test labels and masks are used only for final evaluation.

We report image AUROC/AP, pixel AUROC/AP, image precision/recall/F1 at the validation threshold, and AUPRO up to 30% pixel FPR. AUROC/AP and AUPRO are evaluation measures for the anomaly-detection task; they are not PCA concepts from the lecture.

Masks are resized to the model's 256×256 output grid using max pooling so that very small defect regions are not silently erased by nearest-neighbour downsampling. This can expand a defect boundary by up to a pooling cell, so pixel metrics must be interpreted at this evaluation resolution. Use the same mask-resizing setting for other methods when comparing pixel metrics.

In [ ]:
def collect_outputs(dataset, model):
    image_labels = []
    image_scores = []
    masks = []
    anomaly_maps = []

    for index in range(len(dataset)):
        item = dataset[index]
        image = item["image"].unsqueeze(0).cpu().numpy()
        anomaly_map = model.anomaly_map(image)[0]
        mask = item["mask"].squeeze(0).cpu().numpy().astype(bool)
        label_value = int(item["label"])

        if label_value == 1 and not mask.any():
            raise ValueError(
                f"An anomalous test mask became empty after resizing: {item['image_path']}. "
                "Check the mask-resize setting and original mask."
            )
        if label_value == 0 and mask.any():
            raise ValueError(f"A normal test image has a non-empty mask: {item['image_path']}")

        image_labels.append(label_value)
        image_scores.append(float(anomaly_map.max()))
        masks.append(mask)
        anomaly_maps.append(anomaly_map)

    return (
        np.asarray(image_labels, dtype=np.uint8),
        np.asarray(image_scores, dtype=np.float64),
        np.stack(masks),
        np.stack(anomaly_maps)
    )


def compute_aupro(masks, anomaly_maps, max_fpr=0.30, number_of_thresholds=256):
    masks = np.asarray(masks, dtype=bool)
    anomaly_maps = np.asarray(anomaly_maps, dtype=np.float64)

    if masks.shape != anomaly_maps.shape or masks.ndim != 3:
        raise ValueError("Masks and anomaly maps must have matching (N, H, W) shapes.")
    if not 0 < max_fpr <= 1:
        raise ValueError("max_fpr must lie in (0, 1].")
    if number_of_thresholds < 2:
        raise ValueError("At least two thresholds are required.")
    if not np.isfinite(anomaly_maps).all():
        raise ValueError("Anomaly maps contain NaN or infinity.")

    labelled_masks = []
    region_areas = []
    for mask in masks:
        labels, count = connected_components(mask)
        labelled_masks.append(labels)
        if count:
            region_areas.extend(np.bincount(labels.ravel())[1:].tolist())

    negative_pixel_count = int((~masks).sum())
    if not region_areas or negative_pixel_count == 0:
        return float("nan")

    sampled_thresholds = np.quantile(
        anomaly_maps, np.linspace(1.0, 0.0, number_of_thresholds)
    )
    thresholds = np.unique(np.concatenate((
        [np.nextafter(anomaly_maps.max(), np.inf)],
        sampled_thresholds
    )))

    false_positive_rates = [0.0]
    region_overlaps = [0.0]

    for threshold in thresholds:
        total_false_positives = 0
        overlaps = []

        for image_index, labels in enumerate(labelled_masks):
            prediction = anomaly_maps[image_index] >= threshold
            total_false_positives += np.logical_and(prediction, labels == 0).sum()

            if labels.max() > 0:
                predicted_areas = np.bincount(
                    labels[prediction].ravel(), minlength=int(labels.max()) + 1
                )[1:]
                true_areas = np.bincount(labels.ravel())[1:]
                overlaps.extend((predicted_areas / true_areas).tolist())

        false_positive_rate = total_false_positives / negative_pixel_count
        region_overlap = float(np.mean(overlaps)) if overlaps else 0.0

        false_positive_rates.append(float(false_positive_rate))
        region_overlaps.append(region_overlap)

        if false_positive_rate >= max_fpr:
            break

    x_values = np.asarray(false_positive_rates)
    y_values = np.asarray(region_overlaps)
    order = np.argsort(x_values, kind="stable")
    x_values = x_values[order]
    y_values = y_values[order]

    unique_x = np.unique(x_values)
    unique_y = np.asarray([
        y_values[x_values == value].max() for value in unique_x
    ])

    if unique_x[-1] < max_fpr:
        raise RuntimeError("AUPRO thresholds did not reach the requested FPR boundary.")

    end_value = float(np.interp(max_fpr, unique_x, unique_y))
    inside = unique_x < max_fpr
    curve_x = unique_x[inside]
    curve_y = unique_y[inside]

    if len(curve_x) == 0 or curve_x[0] != 0.0:
        curve_x = np.insert(curve_x, 0, 0.0)
        curve_y = np.insert(curve_y, 0, 0.0)

    curve_x = np.append(curve_x, max_fpr)
    curve_y = np.append(curve_y, end_value)

    return float(np.clip(trapezoid(curve_y, curve_x) / max_fpr, 0.0, 1.0))

def evaluate_model(model, validation_dataset, test_dataset, category_name, method_name):
    validation_labels, validation_scores, _, _ = collect_outputs(validation_dataset, model)
    if len(validation_labels) == 0 or np.any(validation_labels != 0):
        raise ValueError("Validation data must contain normal images only.")

    image_threshold = float(np.quantile(validation_scores, 0.95, method="higher"))

    test_labels, test_scores, test_masks, test_maps = collect_outputs(test_dataset, model)
    if np.unique(test_labels).size != 2:
        raise ValueError(f"The test split for {category_name} must contain both classes.")

    flat_masks = test_masks.reshape(-1).astype(np.uint8)
    if np.unique(flat_masks).size != 2:
        raise ValueError(
            f"Pixel masks for {category_name} contain only one class after resizing."
        )

    image_predictions = (test_scores >= image_threshold).astype(np.uint8)
    flat_scores = test_maps.reshape(-1)

    results = {
        "category": category_name,
        "method": method_name,
        "image_auroc": float(roc_auc_score(test_labels, test_scores)),
        "image_ap": float(average_precision_score(test_labels, test_scores)),
        "image_precision": float(precision_score(test_labels, image_predictions, zero_division=0)),
        "image_recall": float(recall_score(test_labels, image_predictions, zero_division=0)),
        "image_f1": float(f1_score(test_labels, image_predictions, zero_division=0)),
        "pixel_auroc": float(roc_auc_score(flat_masks, flat_scores)),
        "pixel_ap": float(average_precision_score(flat_masks, flat_scores)),
        "aupro_30pct_fpr": compute_aupro(test_masks, test_maps),
        "validation_threshold": image_threshold,
        "validation_images": len(validation_dataset),
        "test_images": len(test_dataset)
    }

    return pd.DataFrame([results]), test_maps, test_masks


def save_result_tables(category_results, method_name):
    results_table = pd.concat(category_results, ignore_index=True)
    metric_names = [
        "image_auroc", "image_ap", "image_precision", "image_recall",
        "image_f1", "pixel_auroc", "pixel_ap", "aupro_30pct_fpr"
    ]

    macro_values = results_table[metric_names].mean()
    macro_table = pd.DataFrame([{"method": method_name, "category": "MACRO", **macro_values.to_dict()}])

    output_directory = Path("results/tables")
    output_directory.mkdir(parents=True, exist_ok=True)

    category_path = output_directory / "global_pca_categories.csv"
    macro_path = output_directory / "global_pca_macro.csv"

    results_table.to_csv(category_path, index=False)
    macro_table.to_csv(macro_path, index=False)

    print("Category results saved to:", category_path)
    print("Macro results saved to:", macro_path)

    return results_table, macro_table

toy_mask = np.zeros((2, 16, 16), dtype=bool)
toy_mask[0, 6:10, 6:10] = True
perfect_map = np.zeros((2, 16, 16), dtype=np.float32)
perfect_map[toy_mask] = 1.0
reversed_map = 1.0 - perfect_map

assert np.isclose(compute_aupro(toy_mask, perfect_map), 1.0)
assert np.isclose(compute_aupro(toy_mask, reversed_map), 0.0)

print("AUPRO synthetic checks passed.")

## 5. Run the selected categories

With RUN_ALL_CATEGORIES=False, this runs bottle only and provides a quick execution check. Set it to True in the setup cell to run all 15 categories. Each category's fitted model is saved separately.

In [ ]:
RETAINED_VARIANCE = float(CONFIG["pca"].get("n_components", 0.95))
CONFIG["data"]["mask_resize"] = "max"

category_results = []
first_preview = None

for category_name in CATEGORIES_TO_RUN:
    print("\nRunning Global PCA for:", category_name)
    start_time = time.perf_counter()

    train_dataset = build_dataset(CONFIG, "pca", "train", category_name)
    validation_dataset = build_dataset(CONFIG, "pca", "val", category_name)
    test_dataset = build_dataset(CONFIG, "pca", "test", category_name)

    training_images = stack_images(train_dataset).cpu().numpy()

    if not len(train_dataset) or not len(validation_dataset) or not len(test_dataset):
        raise ValueError(f"An empty split was found for {category_name}.")
    if not all(sample[2] == "good" for sample in train_dataset.samples):
        raise ValueError("PCA training data must contain only normal images.")
    if not all(sample[2] == "good" for sample in validation_dataset.samples):
        raise ValueError("PCA validation data must contain only normal images.")

    model = GlobalPCA(
        n_components=RETAINED_VARIANCE,
        method="svd",
        ddof=0
    ).fit(training_images)

    metrics, test_maps, test_masks = evaluate_model(
        model,
        validation_dataset,
        test_dataset,
        category_name,
        "Global PCA"
    )

    elapsed_seconds = time.perf_counter() - start_time
    metrics["runtime_seconds"] = elapsed_seconds
    metrics["training_images"] = len(train_dataset)
    metrics["validation_images"] = len(validation_dataset)
    metrics["test_images"] = len(test_dataset)
    metrics["components"] = model.n_components_
    metrics["retained_variance"] = float(model.explained_variance_ratio_.sum())

    model_directory = Path("experiments/pca/global")
    model_directory.mkdir(parents=True, exist_ok=True)
    model.save(model_directory / f"{category_name}.npz")

    category_results.append(metrics)
    print(metrics[[
        "category", "image_auroc", "pixel_auroc", "pixel_ap",
        "aupro_30pct_fpr", "image_f1", "runtime_seconds"
    ]].round(4).to_string(index=False))

    if first_preview is None:
        example = validation_dataset[0]
        example_image = example["image"].unsqueeze(0).cpu().numpy()
        first_preview = (
            example_image[0, 0],
            model.reconstruct(example_image)[0, 0],
            model.anomaly_map(example_image)[0]
        )

category_table, macro_table = save_result_tables(category_results, "Global PCA")
display(category_table.round(4))
display(macro_table.round(4))

if first_preview is not None:
    original_image, reconstructed_image, error_map = first_preview
    figure, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(original_image, cmap="gray")
    axes[0].set_title("Original")
    axes[1].imshow(reconstructed_image, cmap="gray")
    axes[1].set_title("PCA reconstruction")
    axes[2].imshow(error_map, cmap="inferno")
    axes[2].set_title("Anomaly map")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Interpretation and limitations

The mean, centring, covariance, eigen-directions, retained-variance subspace, projection and reconstruction follow the PCA lecture. The economy SVD is the computational route used to obtain the same covariance eigenvectors without allocating the full pixel covariance matrix.

The squared residual map, maximum image score, validation-derived threshold and anomaly-detection metrics are adaptations for this project. They are not claimed as equations taught in the PCA lecture.

Global PCA models normal appearance variation across entire images. It does not model local texture independently, so defects can be missed when they resemble normal global variation. A one-category smoke test is not a dataset-wide result; enable all-category mode before reporting macro performance.